In [1]:
import numpy as np
import pandas as pd
import itertools
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
dataframe=pd.read_csv("news.csv")

In [3]:
dataframe.shape

(6335, 4)

In [4]:
dataframe.tail()

,Unnamed: 0,title,text,label
6330,4490,State Department says it can't find emails fro...,The State Department told the Republican Natio...,REAL
6331,8062,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,FAKE
6332,8622,Anti-Trump Protesters Are Tools of the Oligarc...,Anti-Trump Protesters Are Tools of the Oligar...,FAKE
6333,4021,"In Ethiopia, Obama seeks progress on peace, se...","ADDIS ABABA, Ethiopia —President Obama convene...",REAL
6334,4330,Jeb Bush Is Suddenly Attacking Trump. Here's W...,Jeb Bush Is Suddenly Attacking Trump. Here's W...,REAL


In [5]:
labels=dataframe.label

In [6]:
labels.head()

0    FAKE
1    FAKE
2    REAL
3    FAKE
4    REAL
Name: label, dtype: object

In [7]:
x_train,x_test,y_train,y_test=train_test_split(dataframe["text"], labels, test_size=0.2, random_state=7)

In [8]:
tfidf_vectorizer=TfidfVectorizer(stop_words='english', max_df=0.7)
tfidf_train=tfidf_vectorizer.fit_transform(x_train) 
tfidf_test=tfidf_vectorizer.transform(x_test)

In [9]:
from sklearn import metrics
pac=PassiveAggressiveClassifier(max_iter=50)
pac.fit(tfidf_train,y_train)
y_prte=pac.predict(tfidf_train)
y_pred=pac.predict(tfidf_test)
pa_train = metrics.accuracy_score(y_train,y_prte)
score=accuracy_score(y_test,y_pred)
print(f'Accuracy Training: {round(pa_train*100,2)}%')
print(f'Accuracy Testing: {round(score*100,2)}%')

Accuracy Training: 100.0%
Accuracy Testing: 92.66%


In [10]:
confusion_matrix(y_test,y_pred, labels=['FAKE','REAL'])

array([[588,  50],
       [ 43, 586]], dtype=int64)

In [11]:
from sklearn import metrics
from sklearn.neighbors import KNeighborsClassifier     
knn = KNeighborsClassifier()
knn = knn.fit(tfidf_train, y_train)
knny_pred = knn.predict(tfidf_train)
knny_tst=knn.predict(tfidf_test)
knnac_train = metrics.accuracy_score(y_train,knny_pred)
print('Training Accuracy is: ',knnac_train*100)
knnac_test = metrics.accuracy_score(y_test,knny_tst)
print('Test Accuracy is: ',knnac_test*100)

Training Accuracy is:  59.35280189423836
Test Accuracy is:  56.11681136543015


In [12]:
from sklearn.linear_model import LogisticRegression
logreg=LogisticRegression()
logreg.fit(tfidf_train,y_train)
y_pred = logreg.predict(tfidf_train)
y_tst=logreg.predict(tfidf_test)
train_ac=metrics.accuracy_score(y_train,y_pred)
print('Training Accuracy is: ',train_ac*100)
test_ac=metrics.accuracy_score(y_test,y_tst)
print('Training Accuracy is: ',test_ac*100)

Training Accuracy is:  95.77742699289661
Training Accuracy is:  91.71270718232044


In [13]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf = rf.fit(tfidf_train,y_train)
rfy_pred=rf.predict(tfidf_train)
rfy_test=rf.predict(tfidf_test)
rfac_train = metrics.accuracy_score(y_train,rfy_pred)
print('Training Accuracy is: ',rfac_train*100)
rfac_test= metrics.accuracy_score(y_test,rfy_test)
print('Test Accuracy is: ',rfac_test*100)

Training Accuracy is:  100.0
Test Accuracy is:  90.13417521704814


In [14]:
from sklearn.svm import SVC
svm = SVC()
svm = svm.fit(tfidf_train, y_train)
svmy_pred = svm.predict(tfidf_train)
svmy_tst=svm.predict(tfidf_test)
svmac_train= metrics.accuracy_score(y_train,svmy_pred )
print('Training Accuracy is: ',svmac_train)
svmac_test = metrics.accuracy_score(y_test,svmy_tst)
print('Test Accuracy is: ',svmac_test)

Training Accuracy is:  0.9978295185477506
Test Accuracy is:  0.9289660615627466


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
bigram_vectorizer = CountVectorizer(analyzer = "word", ngram_range=(1,2))
tf_idf = TfidfTransformer(norm="l2")
pipeline = Pipeline([
     ('bigram_vectorizer', bigram_vectorizer),
     ('tfidf', tf_idf),
     ('clf', PassiveAggressiveClassifier()),
 ])

In [16]:
pipeline.fit(x_train,y_train)

Pipeline(steps=[('bigram_vectorizer', CountVectorizer(ngram_range=(1, 2))),
                ('tfidf', TfidfTransformer()),
                ('clf', PassiveAggressiveClassifier())])

In [17]:
predictPipeline=pipeline.predict(x_test)
model=accuracy_score(y_test,predictPipeline)
print(model)

0.9344909234411997


In [18]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
trigram_vectorizer = CountVectorizer(analyzer = "word", ngram_range=(1,3))
tf_idf = TfidfTransformer(norm="l2")
pipeline = Pipeline([
     ('trigram_vectorizer', trigram_vectorizer),
     ('tfidf', tf_idf),
     ('clf', PassiveAggressiveClassifier()),
 ])
pipeline.fit(x_train,y_train)
predictPipeline=pipeline.predict(x_test)
model=accuracy_score(y_test,predictPipeline)
print(model)

0.9313338595106551


In [19]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
trigram_vectorizer = CountVectorizer(analyzer = "word", ngram_range=(1,2))
tf_idf = TfidfTransformer(norm="l2")
pipeline = Pipeline([
     ('trigram_vectorizer', trigram_vectorizer),
     ('tfidf', tf_idf),
     ('clf', LogisticRegression()),
 ])
pipeline.fit(x_train,y_train)
predictPipeline=pipeline.predict(x_test)
model=accuracy_score(y_test,predictPipeline)
print(model)

0.9155485398579322


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
trigram_vectorizer = CountVectorizer(analyzer = "word", ngram_range=(1,3))
tf_idf = TfidfTransformer(norm="l2")
pipeline1 = Pipeline([
     ('trigram_vectorizer', trigram_vectorizer),
     ('tfidf', tf_idf),
     ('clf', PassiveAggressiveClassifier()),
 ])
pipeline1.fit(x_train,y_train)
predictPipeline=pipeline1.predict(data)

In [ ]:
print(predictPipeline)